
# Practice 1 - Geometry and Topology in Machine Learning

We will now explore some of the concepts of the lecture in Python. This notebook is divided into four sections:

1. **Warm-up**: tensors, autograd, and gradient descent by hand
2. **An MLP classifier**: the multilayer perceptron on a non-linearly-separable dataset
3. **Opening up the training loop**: learning rate, batch size, over/underfitting
4. **From filters to a CNN**: hand-crafted convolutions, then LeNet on MNIST

Cells marked **`# TODO`** are for you to fill in. A solutions notebook will be provided soon in the course website.


## 0. Setup

In [ ]:
# importing libraries
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# setting randomness for reproducibility
torch.manual_seed(0)
np.random.seed(0)

# checking if we have access to GPUs with CUDA, otherwise default to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)


## 1. Warm-up: tensors, autograd, gradient descent

We have seen that a **perceptron** computes $\hat{y} = g(\mathbf{w}\cdot\mathbf{x} + b)$.

PyTorch tensors behave like NumPy arrays, but if we set `requires_grad=True`, PyTorch records every operation so it can compute derivatives for us in **backpropagation**. 

We code the following linear operation
$$z = \begin{pmatrix}1.0 & 2.0 & -0.5\end{pmatrix} \cdot \begin{pmatrix}0.5 \\ -1.0 \\ 2.0\end{pmatrix} + 0.0 $$
and the corresponding non-linearity (a sigmoid function) applied to it to give the predicted $\hat{y}$.


In [ ]:
# One perceptron, evaluated by hand.
x = torch.tensor([0.5, -1.0, 2.0])
w = torch.tensor([1.0, 2.0, -0.5], requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

z = w @ x + b                 # weighted sum
y_hat = torch.sigmoid(z)      # non-linearity g
print("z =", z.item(), " y_hat =", y_hat.item())

# Pretend the target is 1.0 and use a squared error "loss".
loss = (y_hat - 1.0) ** 2
loss.backward()               # fills w.grad and b.grad via the chain rule

print("dL/dw =", w.grad)
print("dL/db =", b.grad)


### Gradient descent by hand

Let's fit a line $y = w x + b$ to noisy data by *manually* running the update
rule from slide 27,

$$\theta \leftarrow \theta - \eta\, \frac{\partial L}{\partial \theta},$$

with the mean-squared-error loss. We never write the derivatives
ourselves, `loss.backward()` does it.


In [ ]:
# Synthetic data: y = 2x + 1 + noise
N = 100
x_data = torch.linspace(-3, 3, N)
y_data = 2.0 * x_data + 1.0 + 0.7 * torch.randn(N)

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.05
history = []

for step in range(60):
    y_pred = w * x_data + b
    loss = ((y_pred - y_data) ** 2).mean()     # MSE

    # TODO(1): run one gradient-descent step.
    #   (a) call loss.backward()
    #   (b) inside `with torch.no_grad():` update  w <- w - lr * w.grad  (same for b)
    #   (c) zero w.grad and b.grad so they don't accumulate
    ...

    history.append(loss.item())

print(f"learned  w = {w.item():.3f}   b = {b.item():.3f}   (true: 2.0, 1.0)")

# plot the results
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(x_data, y_data, s=12, alpha=0.6)
xs = torch.linspace(-3, 3, 2)
ax[0].plot(xs, (w * xs + b).detach(), "r-", lw=2, label="fit")
ax[0].set_title("data + fitted line"); ax[0].legend()
ax[1].plot(history); ax[1].set_xlabel("step"); ax[1].set_ylabel("MSE loss")
ax[1].set_title("loss curve"); plt.show()


## 2. An MLP classifier

We now use the real building blocks from `torch` to build an MLP: `nn.Linear` layers stacked with
non-linear activations. The function `nn.Sequential()` stacks these different layers.

We test it with the same dataset from slides 18 and 19: a set of points in $\mathbb{R}^2$ with the shape of two interlaced moons.


In [ ]:
from sklearn.datasets import make_moons

# sample data from built-in function from sklearn
X_np, y_np = make_moons(n_samples=1000, noise=0.20, random_state=0)
X_np = (X_np - X_np.mean(0)) / X_np.std(0)          # standardize

# turn the data into tensors to handle with torch
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

# train / test split
perm = torch.randperm(len(X))
tr, te = perm[:800], perm[800:]
X_tr, y_tr, X_te, y_te = X[tr], y[tr], X[te], y[te]

# visualize dataset
plt.figure(figsize=(5, 5))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="viridis", s=12, edgecolor="k", lw=0.2)
plt.title("two moons"); plt.show()

In [ ]:
def make_mlp(hidden=16, activation=nn.ReLU):
    torch.manual_seed(0)
    # TODO(2a): build a 3-layer MLP with nn.Sequential:
    #   Linear(2 -> hidden) , activation() , Linear(hidden -> hidden) ,
    #   activation() , Linear(hidden -> 1)
    # The last layer outputs ONE raw logit (no sigmoid).
    return ...

model = make_mlp().to(device)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
def train_clf(model, X_tr, y_tr, X_te, y_te, lr=0.5, epochs=200, batch_size=None):
    model = model.to(device)
    Xtr, ytr, Xte, yte = X_tr.to(device), y_tr.to(device), X_te.to(device), y_te.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)     # use gradient descent on the whole data
    loss_fn = nn.BCEWithLogitsLoss()          # sigmoid + cross-entropy in one, slide 11
    log = {"train_loss": [], "test_loss": [], "test_acc": []}

    for epoch in range(epochs):
        idx = torch.randperm(len(Xtr))
        bs = batch_size or len(Xtr)
        for i in range(0, len(Xtr), bs):
            b = idx[i:i + bs]
            logits = model(Xtr[b])
            loss = loss_fn(logits, ytr[b])

            # TODO(2b): the four lines at the heart of every PyTorch training loop
            #   1. opt.zero_grad()   2. loss.backward()   3. opt.step()
            ...

        with torch.no_grad():
            log["train_loss"].append(loss_fn(model(Xtr), ytr).item())
            test_logits = model(Xte)
            log["test_loss"].append(loss_fn(test_logits, yte).item())
            pred = (test_logits > 0).float()
            log["test_acc"].append((pred == yte).float().mean().item())
    return log

log = train_clf(make_mlp(), X_tr, y_tr, X_te, y_te)
print(f"final test accuracy: {log['test_acc'][-1]:.3f}")

In [ ]:
def plot_boundary(model, title=""):
    model = model.to("cpu").eval()
    xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 300), np.linspace(-2.5, 2.5, 300))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        probs = torch.sigmoid(model(grid)).reshape(xx.shape)
    plt.figure(figsize=(5, 5))
    plt.contourf(xx, yy, probs, levels=20, cmap="coolwarm", alpha=0.7)
    plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", s=10, edgecolor="k", lw=0.2)
    plt.title(title); plt.show()

m = make_mlp()
train_clf(m, X_tr, y_tr, X_te, y_te)
plot_boundary(m, "MLP with ReLU learns a curved boundary")


### Why the non-linearity matters

Replace every activation with the identity. A stack of linear layers is *still
just a linear map*, so it cannot bend the decision boundary around the moons.


In [ ]:
m_lin = make_mlp(activation=nn.Identity)
log_lin = train_clf(m_lin, X_tr, y_tr, X_te, y_te)
print(f"linear model test accuracy: {log_lin['test_acc'][-1]:.3f}")
plot_boundary(m_lin, "No non-linearity, boundary is a straight line")

**Try it:** swap in `nn.Tanh` or `nn.GELU` (slide 17) and re-plot. Does the boundary change?

In [ ]:
for act in (nn.Tanh, nn.GELU):
    m = make_mlp(activation=act)
    lg = train_clf(m, X_tr, y_tr, X_te, y_te)
    print(f"{act.__name__:8s} test accuracy: {lg['test_acc'][-1]:.3f}")
    plot_boundary(m, f"activation = {act.__name__}")


## 3. Opening up the training loop

### 3.1 Learning rate

The learning rate $\eta$ scales every step. If it is too small, it moves too slow and it makes it hard to progress towards the minimum; if it's too large, it
overshoots and diverges.


In [ ]:
plt.figure(figsize=(7, 4))
for lr in (1e-2, 5e-1, 1e1):
    # TODO(3): train a fresh make_mlp() with this lr for 150 epochs, then plot the "train_loss"
    ...
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("train loss (log)")
plt.legend(); plt.title("effect of the learning rate"); plt.show()


### 3.2 Batch size 

Full-batch gradient descent uses all $n$ points per step (smooth, expensive).
Mini-batch SGD uses $B$ points (noisy, cheap, and the noise can help). If $B$ is too small, the noise dominates and we don't converge.


In [ ]:
plt.figure(figsize=(7, 4))
for bs in (None, 128, 64, 32):
    lg = train_clf(make_mlp(), X_tr, y_tr, X_te, y_te, lr=5e-1, epochs=100, batch_size=bs)
    plt.plot(lg["train_loss"], label=f"batch_size = {bs or len(X_tr)}")
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.legend()
plt.title("full-batch vs mini-batch"); plt.show()


### 3.3 Overfitting 

Give a large model only a handful of training points. The training loss keeps
dropping while the test loss turns back up because the model memorizes noise.


In [ ]:
small_tr = perm[:30] # small training data
Xs, ys = X[small_tr], y[small_tr]

log_of = train_clf(make_mlp(hidden=128), Xs, ys, X_te, y_te, lr=5e-1, epochs=400) # choosing a big number of parameters
plt.figure(figsize=(7, 4))
plt.plot(log_of["train_loss"], label="train (30 points)")
plt.plot(log_of["test_loss"], label="test (200 points)")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("overfitting: test loss diverges from train loss"); plt.show()


## 4. From filters to a CNN

### 4.1 Convolution is just a sliding weighted sum

Before *learning* filters, let's apply a few classic hand-designed kernels with
`F.conv2d`, exactly as in the lecture (Gaussian blur, sharpening, Sobel edges).


In [ ]:
try:
    from skimage.data import camera
    img = camera().astype(np.float32) / 255.0
except Exception:
    yy, xx = np.mgrid[0:256, 0:256]
    img = ((np.sin(xx / 8) * np.cos(yy / 12) > 0).astype(np.float32))
    img[64:192, 64:192] = 1.0

x = torch.tensor(img)[None, None]        # shape (1, 1, H, W)

def gaussian_kernel(k=7, sigma=1.7):
    ax = torch.arange(k) - k // 2
    g = torch.exp(-(ax[:, None] ** 2 + ax[None, :] ** 2) / (2 * sigma ** 2))
    return (g / g.sum())[None, None]

K_blur  = gaussian_kernel()
# TODO(4a): define the 3x3 sharpening kernel (slide 46) and the 3x3 Sobel-x kernel,
#           each shaped (1, 1, 3, 3).
K_sharp = ...
K_sobel = ...

# TODO(4b): apply each kernel with F.conv2d(x, K, padding=...) — pick padding so the
#           output keeps the same H, W.
outs = {
    "original": x,
    "Gaussian blur": F.conv2d(x, K_blur, padding=3),
    "sharpen": ...,
    "Sobel-x (edges)": ...,
}
fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a, (name, o) in zip(ax, outs.items()):
    a.imshow(o[0, 0].numpy(), cmap="gray"); a.set_title(name); a.axis("off")
plt.show()


### 4.2 Convolution is translation-equivariant

Shifting the image and then convolving gives (away from the borders) the same
result as convolving and then shifting: $T(I) \circledast K = T(I \circledast K)$.


In [ ]:
shift = (25, 25)
x_shift = torch.roll(x, shifts=shift, dims=(2, 3))

plt.title("image after translating 25 columns and 25 rows")
plt.imshow(x_shift[0, 0].numpy(), cmap="gray"); a.set_title(name) 
plt.axis("off")

conv_then_shift = torch.roll(F.conv2d(x, K_sobel, padding=1), shifts=shift, dims=(2, 3))
shift_then_conv = F.conv2d(x_shift, K_sobel, padding=1)

# compare on the interior (ignore wrapped border)
inter = (slice(None), slice(None), slice(40, -40), slice(40, -40))
diff = (conv_then_shift[inter] - shift_then_conv[inter]).abs().max()
print(f"max interior difference: {diff.item():.2e}")


### 4.3 MLP vs CNN on MNIST 

An MLP on images ignores spatial structure and needs huge weight matrices.
A CNN shares a small kernel across the whole image, which implies fewer parameters, and has been shown to have better
accuracy. We use a subset of MNIST so it trains quickly.

*(For this section, `Runtime ▸ Change runtime type ▸ GPU` in Colab makes training ~10× faster.)*


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

tf = transforms.ToTensor()
train_full = datasets.MNIST("./data", train=True,  download=True, transform=tf)
test_full  = datasets.MNIST("./data", train=False, download=True, transform=tf)

train_ds = Subset(train_full, range(20000))
test_ds  = Subset(test_full,  range(5000))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=512)

imgs, labels = next(iter(train_dl))
fig, ax = plt.subplots(1, 8, figsize=(12, 2))
for a, im, lb in zip(ax, imgs, labels):
    a.imshow(im[0], cmap="gray"); a.set_title(int(lb)); a.axis("off")
plt.show()

In [ ]:
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128), nn.ReLU(),
    nn.Linear(128, 10),
)

class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)   # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(6, 16, 5)             # 14x14 -> 10x10
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # TODO(5): implement the LeNet forward pass (slide 49):
        #   conv1 -> ReLU -> 2x2 max-pool   (28 -> 14)
        #   conv2 -> ReLU -> 2x2 max-pool   (10 -> 5)
        #   flatten -> fc1 -> ReLU -> fc2 -> ReLU -> fc3
        # use F.relu and F.max_pool2d(..., 2)
        ...

def n_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"MLP   parameters: {n_params(mlp):,}")
print(f"LeNet parameters: {n_params(LeNet()):,}")

In [ ]:
def train_mnist(model, epochs=5, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()           # softmax + cross-entropy, multi-class

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)

            # TODO(6): the standard loop — forward, loss, zero_grad, backward, step
            ...

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in test_dl:
                xb, yb = xb.to(device), yb.to(device)
                correct += (model(xb).argmax(1) == yb).sum().item()
                total += len(yb)
        print(f"  epoch {epoch + 1}: test accuracy = {correct / total:.4f}")
    return correct / total

print("MLP:")
acc_mlp = train_mnist(mlp)
print("LeNet:")
acc_cnn = train_mnist(LeNet())
print(f"\nMLP  {acc_mlp:.4f}   |   LeNet {acc_cnn:.4f}")

### 4.4 What did the first convolutional layer learn?

In [ ]:
trained = LeNet()
train_mnist(trained, epochs=1)
W = trained.conv1.weight.detach().cpu()      # shape (6, 1, 5, 5)

fig, ax = plt.subplots(1, 6, figsize=(12, 2.5))
for i, a in enumerate(ax):
    a.imshow(W[i, 0], cmap="gray"); a.axis("off"); a.set_title(f"filter {i}")
plt.suptitle("learned 5x5 filters of conv1"); plt.show()


## Optional: self-attention 

Implement one attention head on the toy sentence and visualize the attention
matrix. No training, just the formula
$\operatorname{softmax}\!\big(\tfrac{QK^\top}{\sqrt{d'}}\big)V$.


In [ ]:
# OPTIONAL: self-attention from scratch 
tokens = ["The", "cat", "ate", "the", "mouse"]
T, d, d_head = len(tokens), 8, 4
torch.manual_seed(0)

X_emb = torch.randn(T, d)                      # pretend token embeddings
W_Q, W_K, W_V = (torch.randn(d, d_head) for _ in range(3))

Q, K, V = X_emb @ W_Q, X_emb @ W_K, X_emb @ W_V
scores = Q @ K.T / d_head ** 0.5
A = torch.softmax(scores, dim=-1)             # (T, T) attention weights
out = A @ V

plt.figure(figsize=(4.5, 4))
plt.imshow(A.detach(), cmap="viridis")
plt.xticks(range(T), tokens, rotation=45); plt.yticks(range(T), tokens)
plt.colorbar(); plt.title("attention weights  A = softmax(QKᵀ/√d')"); plt.show()